# Bài 05 — Xử lý danh bạ Excel (2a, 2b, 2c)
**Cách chạy:** tải notebook này lên Google Colab → Runtime → Run all.

- Mặc định đọc CSV từ Google Sheets theo `sheet_id` và `gid` đã cung cấp. Sửa `SOURCE` thành `"sample"` để chạy với 8 người trong file `contacts.xlsx` mẫu đã nhúng sẵn.
- Bản gốc `data.xlsx` cùng thư mục bài tập được nhúng trong notebook để giữ các sheet khác. Muốn dùng file khác: tải `data.xlsx` lên thanh Files của Colab trước khi chạy. Notebook không ghi đè file đầu vào; kết quả nằm trong `output/`.
- Kết quả: `output/data.xlsx` (2a–2b), `output/contact.xlsx` (2c), và file ZIP để tải một lần.
- Các file Excel đính kèm chỉ được dùng làm mẫu dữ liệu/kỹ thuật, không được coi là chỉ dẫn bổ sung. Áp dụng `load_workbook`, định dạng ô, `BarChart` và `Reference` tương tự bài openpyxl.
- Tuổi là số năm đã tròn tại ngày chốt, không lấy tuổi cũ trong file mẫu. Mặc định ngày chốt theo múi giờ Việt Nam.

In [ ]:
%pip -q install pandas openpyxl pillow

## 1. Thư viện và cấu hình
URL phải là chuỗi URL thuần, không chứa cú pháp liên kết Markdown `[...](...)`.
`dtype=str` giúp giữ số 0 đầu của số điện thoại. Nếu Sheets đã lưu điện thoại dạng số và mất số 0, chương trình không tự suy đoán để thêm lại.

In [ ]:
from pathlib import Path
from datetime import date, datetime
from zoneinfo import ZoneInfo
from io import BytesIO, StringIO
import base64, re, unicodedata, urllib.request, zipfile
from collections import Counter
import pandas as pd
from IPython.display import display
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.chart import BarChart, Reference
from openpyxl.chart.label import DataLabelList
from openpyxl.utils import get_column_letter

SOURCE = "google"  # "google" hoặc "sample"
SHEET_ID = "1Jw4JavfSza6F9tGh0xFRNnRrLPHSFYk6"
GID = "1498967871"
URL = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}"
AS_OF = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date()
# Ví dụ ngày chốt cố định: AS_OF = date(2023, 1, 31)
INPUT_DATA = Path("data.xlsx")
OUT = Path("output")
OUT.mkdir(exist_ok=True)
print("Ngày tính tuổi:", AS_OF.strftime("%d-%m-%Y"))

## 2. Tài nguyên mẫu đi kèm
Cell dưới chứa bản sao nguyên vẹn của hai file mẫu, vì vậy không cần tải lại chúng lên Colab. Có thể thu gọn cell này.

In [ ]:
TEMPLATE_B64 = '...'
SAMPLE_B64 = '...'

## 3. Đọc dữ liệu và kiểm tra cột
Tự tìm hàng tiêu đề kể cả khi bảng có dòng trống phía trên. Cần các cột **Họ và tên, Nơi sinh, Ngày sinh**; giữ thêm các cột có trong nguồn. Nếu link không truy cập được, cell dừng với hướng dẫn rõ ràng, không âm thầm đổi nguồn.

In [ ]:
def text(value):
    return unicodedata.normalize("NFC", " ".join(str(value).strip().split()))

def simple(value):
    value = unicodedata.normalize("NFD", text(value).lower().replace("đ", "d"))
    return "".join(c for c in value if not unicodedata.combining(c))

ALIASES = {
    "ho va ten": "Họ và tên", "ho ten": "Họ và tên",
    "ngay sinh": "Ngày sinh", "noi sinh": "Nơi sinh",
    "sdt": "SĐT", "so dien thoai": "SĐT", "dien thoai": "SĐT",
    "gioi tinh": "Giới tính", "email": "email", "stt": "STT",
    "age": "Tuổi", "tuoi": "Tuổi",
}
REQUIRED = {"Họ và tên", "Nơi sinh", "Ngày sinh"}

def canonical(v):
    return ALIASES.get(simple(v), text(v))

def find_table(rows):
    for i, row in enumerate(rows):
        names = [canonical(v) if v is not None else "" for v in row]
        if REQUIRED.issubset(names):
            indices = [j for j, n in enumerate(names) if n and not n.startswith("Unnamed:")]
            headers = [names[j] for j in indices]
            if len(headers) != len(set(headers)):
                raise ValueError("Có tiêu đề cột trùng nhau; hãy kiểm tra nguồn.")
            body = [[r[j] if j < len(r) and r[j] is not None else "" for j in indices]
                    for r in rows[i + 1:] if any(v is not None and str(v).strip() for v in r)]
            return pd.DataFrame(body, columns=headers)
    raise ValueError("Không tìm được bảng có các cột Họ và tên, Nơi sinh, Ngày sinh. Kiểm tra gid.")

if SOURCE == "google":
    try:
        with urllib.request.urlopen(URL, timeout=45) as response:
            csv_text = response.read().decode("utf-8-sig")
        if "<html" in csv_text[:500].lower() or "<!doctype" in csv_text[:500].lower():
            raise ValueError("Link trả về HTML thay vì CSV.")
        raw = pd.read_csv(StringIO(csv_text), header=None, dtype=str, keep_default_na=False)
        contacts = find_table(raw.values.tolist())
    except Exception as exc:
        raise RuntimeError(
            "Không đọc được Google Sheets. Kiểm tra sheet_id/gid và quyền đọc công khai. "
            "Để thử bằng Excel mẫu, đặt SOURCE = 'sample' ở cell 1 rồi chạy lại. "
            f"Chi tiết: {exc}"
        ) from exc
elif SOURCE == "sample":
    sample_wb = load_workbook(BytesIO(base64.b64decode(SAMPLE_B64)), data_only=True)
    contacts = pd.concat([find_table(list(ws.values)) for ws in sample_wb], ignore_index=True)
else:
    raise ValueError("SOURCE chỉ nhận 'google' hoặc 'sample'.")

if contacts.empty:
    raise ValueError("Nguồn dữ liệu không có người nào.")
print(f"Nguồn: {SOURCE} | Số người: {len(contacts)}")
display(contacts.head())

## 4. Câu 2a — Chuẩn hóa, tính tuổi và sắp xếp
Quy ước tách tên: từ cuối là **tên**, từ đầu là **họ**, phần giữa là **tên đệm**. Sắp xếp theo thứ tự `(tên, họ, tên đệm)` với bảng chữ cái tiếng Việt (Đ sau D; Ă, Â sau A…). Các tên trùng hoàn toàn giữ thứ tự nguồn. Họ kép được xử lý theo quy ước từ đầu nêu trên.

Tuổi = năm chốt − năm sinh − 1 nếu chưa đến sinh nhật. Người sinh 29/02 được tính qua sinh nhật vào 01/03 trong năm không nhuận. Ngày sinh được lưu thành **ngày Excel thật**, chỉ đổi định dạng hiển thị thành `dd-mm-yyyy`.

In [ ]:
ALPHABET = "aăâbcdđeêfghijklmnoôơpqrstuưvwxy z".replace(" ", "")
LETTER_ORDER = {c: i for i, c in enumerate(ALPHABET)}
TONES = {"\u0300": 1, "\u0309": 2, "\u0303": 3, "\u0301": 4, "\u0323": 5}

def vi_key(value):
    letters, tones = [], []
    for char in text(value).casefold():
        parts = unicodedata.normalize("NFD", char)
        tone = next((TONES[c] for c in parts if c in TONES), 0)
        base = unicodedata.normalize("NFC", "".join(c for c in parts if c not in TONES))
        letters.append(LETTER_ORDER.get(base, 1000 + ord(char)))
        tones.append(tone)
    return tuple(letters), tuple(tones)

def name_key(value):
    parts = text(value).split()
    given = parts[-1]
    family = parts[0] if len(parts) > 1 else ""
    middle = " ".join(parts[1:-1])
    return vi_key(given), vi_key(family), vi_key(middle)

def parse_birth(value):
    if isinstance(value, (datetime, date)):
        return value.date() if isinstance(value, datetime) else value
    for fmt in ("%d-%m-%Y", "%d/%m/%Y", "%Y-%m-%d", "%Y-%m-%d %H:%M:%S", "%d.%m.%Y"):
        try:
            return datetime.strptime(text(value), fmt).date()
        except ValueError:
            pass
    raise ValueError(f"Ngày sinh không hợp lệ: {value!r}; hãy dùng dd-mm-yyyy.")

clean = contacts.copy()
for column in clean.columns:
    if column != "Ngày sinh":
        clean[column] = clean[column].map(text)
for column in ["Họ và tên", "Nơi sinh", "Ngày sinh"]:
    if clean[column].map(lambda v: not str(v).strip()).any():
        raise ValueError(f"Có ô trống ở cột {column}. Hãy bổ sung trước khi xử lý.")
clean["Ngày sinh"] = clean["Ngày sinh"].map(parse_birth)
if any(b > AS_OF for b in clean["Ngày sinh"]):
    raise ValueError("Có ngày sinh sau ngày tính tuổi.")
clean["Tuổi"] = clean["Ngày sinh"].map(
    lambda b: AS_OF.year - b.year - ((AS_OF.month, AS_OF.day) < (b.month, b.day)))
order = sorted(clean.index, key=lambda i: name_key(clean.at[i, "Họ và tên"]))
sorted_contacts = clean.loc[order].reset_index(drop=True)
sorted_contacts["STT"] = range(1, len(sorted_contacts) + 1)
columns = ["STT"] + [c for c in sorted_contacts.columns if c not in ("STT", "Tuổi")] + ["Tuổi"]
sorted_contacts = sorted_contacts[columns]
preview = sorted_contacts.copy()
preview["Ngày sinh"] = preview["Ngày sinh"].map(lambda d: d.strftime("%d-%m-%Y"))
display(preview)

## 5. Định dạng bảng bằng openpyxl
Tiêu đề xanh đậm, hàng xen kẽ, đường viền mảnh, bộ lọc, cố định dòng tiêu đề và cột STT. Độ rộng cột tự điều chỉnh; số điện thoại ở dạng text. Ngày và tuổi có định dạng riêng. Mỗi sheet có ghi ngày chốt tuổi để đối chiếu thống kê.

In [ ]:
NAVY, BLUE, PALE = "17365D", "247BA0", "EDF4FA"

def put(ws, row, col, value):
    cell = ws.cell(row, col, value)
    if isinstance(value, str):
        cell.data_type = "s"  # Nội dung nguồn luôn là văn bản, kể cả khi bắt đầu bằng =.
    return cell

def write_table(ws, frame, title, table_name):
    n = len(frame.columns)
    ws.sheet_view.showGridLines = False
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=n)
    put(ws, 1, 1, title)
    ws.cell(1, 1).font = Font(name="Calibri", size=18, bold=True, color=NAVY)
    ws.row_dimensions[1].height = 32
    ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=n)
    put(ws, 2, 1, f"{len(frame)} người • Tuổi tính đến {AS_OF:%d-%m-%Y}")
    ws.cell(2, 1).font = Font(name="Calibri", italic=True, color="64748B")
    for j, col in enumerate(frame.columns, 1):
        cell = put(ws, 4, j, str(col))
        cell.fill = PatternFill("solid", fgColor=NAVY)
        cell.font = Font(name="Calibri", bold=True, color="FFFFFF")
        cell.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[4].height = 28
    for i, row in enumerate(frame.itertuples(index=False, name=None), 5):
        ws.row_dimensions[i].height = 24
        for j, value in enumerate(row, 1):
            cell = put(ws, i, j, value)
            col = frame.columns[j - 1]
            cell.font = Font(name="Calibri", size=11, color=NAVY)
            cell.fill = PatternFill("solid", fgColor=PALE if i % 2 else "FFFFFF")
            cell.border = Border(bottom=Side(style="hair", color="DCE6F1"))
            cell.alignment = Alignment(vertical="center", wrap_text=True,
                horizontal="center" if col in ("STT", "Ngày sinh", "Tuổi", "Giới tính") else "left")
            cell.number_format = "dd-mm-yyyy" if col == "Ngày sinh" else "0" if col in ("STT", "Tuổi") else "@"
    end = len(frame) + 4
    table = Table(displayName=table_name, ref=f"A4:{get_column_letter(n)}{end}")
    table.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
    ws.add_table(table)
    for j, col in enumerate(frame.columns, 1):
        width = max([len(str(col))] + [len(str(v)) for v in frame[col]]) + 3
        ws.column_dimensions[get_column_letter(j)].width = min(42, max(13, width))
    ws.freeze_panes = "B5"
    ws.print_title_rows = "1:4"
    ws.print_options.horizontalCentered = True
    ws.sheet_properties.pageSetUpPr.fitToPage = True
    ws.page_setup.orientation = "landscape"
    ws.page_setup.paperSize = ws.PAPERSIZE_A4
    ws.page_setup.fitToWidth, ws.page_setup.fitToHeight = 1, 0
    ws.print_area = f"A1:{get_column_letter(n)}{end}"

wb = load_workbook(INPUT_DATA) if INPUT_DATA.exists() else load_workbook(BytesIO(base64.b64decode(TEMPLATE_B64)))
kept_names = [s for s in wb.sheetnames if s not in ("sorted_contacts", "statistics")]
kept_values = {s: tuple(wb[s].values) for s in kept_names}
for name in ("sorted_contacts", "statistics"):
    if name in wb.sheetnames:
        del wb[name]
# Tránh trùng tên bảng với các sheet được giữ lại.
used_tables = {name.lower() for ws in wb for name in ws.tables}
table_name = "SortedContacts"
while table_name.lower() in used_tables:
    table_name += "_new"
write_table(wb.create_sheet("sorted_contacts"), sorted_contacts, "DANH BẠ ĐÃ SẮP XẾP", table_name)
print("Giữ lại các sheet:", kept_names)

## 6. Câu 2b — Hai biểu đồ thanh
Thống kê theo **Nơi sinh** và theo **từng tuổi thực tế**, không gom nhóm tuổi. Dữ liệu và biểu đồ cùng nằm trong sheet `statistics`. Đây là biểu đồ Excel có thể chỉnh sửa, được hiển thị khi mở file bằng Excel hoặc phần mềm hỗ trợ biểu đồ XLSX.

In [ ]:
province_counts = sorted_contacts.groupby("Nơi sinh").size()
province_counts = province_counts.reindex(sorted(province_counts.index, key=vi_key))
age_counts = sorted_contacts.groupby("Tuổi").size().sort_index()
ws = wb.create_sheet("statistics")
ws.sheet_view.showGridLines = False
ws.merge_cells("A1:L1")
ws["A1"] = "THỐNG KÊ DANH BẠ"
ws["A1"].font = Font(size=20, bold=True, color=NAVY)
ws.row_dimensions[1].height = 34
ws["A2"] = f"Tổng: {len(sorted_contacts)} người | Ngày chốt: {AS_OF:%d-%m-%Y}"

def statistics_block(start, heading, counts, chart_title, category_title, color):
    put(ws, start, 1, heading)
    put(ws, start, 2, "Số người")
    for col in (1, 2):
        ws.cell(start, col).fill = PatternFill("solid", fgColor=NAVY)
        ws.cell(start, col).font = Font(bold=True, color="FFFFFF")
    for r, (category, count) in enumerate(counts.items(), start + 1):
        put(ws, r, 1, int(category) if heading == "Tuổi" else str(category))
        ws.cell(r, 2, int(count))
        for col in (1, 2):
            ws.cell(r, col).fill = PatternFill("solid", fgColor=PALE if r % 2 else "FFFFFF")
    chart = BarChart()
    chart.type, chart.style = "bar", 10
    chart.title, chart.x_axis.title, chart.y_axis.title = chart_title, "Số người", category_title
    chart.add_data(Reference(ws, min_col=2, min_row=start, max_row=start + len(counts)), titles_from_data=True)
    chart.set_categories(Reference(ws, min_col=1, min_row=start + 1, max_row=start + len(counts)))
    chart.legend = None
    chart.x_axis.scaling.min = 0
    chart.x_axis.majorUnit = max(1, (int(counts.max()) + 9) // 10)
    chart.y_axis.scaling.orientation = "maxMin"
    chart.width, chart.height = 24, max(9, len(counts) * 0.65 + 3)
    chart.series[0].graphicalProperties.solidFill = color
    chart.dataLabels = DataLabelList()
    chart.dataLabels.showVal = True
    ws.add_chart(chart, f"D{start}")
    return start + max(len(counts) + 4, int(chart.height * 3) + 4)

next_row = statistics_block(4, "Nơi sinh", province_counts, "Số người theo tỉnh/thành nơi sinh", "Nơi sinh", BLUE)
statistics_block(next_row, "Tuổi", age_counts, "Số người theo từng độ tuổi", "Tuổi", "E79B35")
ws.column_dimensions["A"].width = 30
ws.column_dimensions["B"].width = 16
ws.column_dimensions["C"].width = 4
ws.freeze_panes = "A5"
wb.save(OUT / "data.xlsx")
display(province_counts.rename("Số người").to_frame())
display(age_counts.rename("Số người").to_frame())

## 7. Câu 2c — Tạo contact.xlsx theo nơi sinh
Mỗi nơi sinh có một sheet, giữ thứ tự tên đã sắp xếp; đánh lại STT trong mỗi sheet. Tên sheet Excel tối đa 31 ký tự và không chứa `[]:*?/\`; tên không hợp lệ được chuẩn hóa và thêm hậu tố nếu trùng.

In [ ]:
def safe_sheet_name(value, used):
    base = re.sub(r"[\[\]:*?/\\]", "_", str(value)).strip().strip("'")[:31] or "Khong_ro"
    if base.lower() == "history":
        base = "History_"
    candidate, suffix = base, 2
    while candidate.casefold() in used:
        tail = f"_{suffix}"
        candidate = base[:31 - len(tail)] + tail
        suffix += 1
    used.add(candidate.casefold())
    return candidate

contact_wb = Workbook()
contact_wb.remove(contact_wb.active)
used, sheet_mapping = set(), {}
for i, province in enumerate(province_counts.index, 1):
    group = sorted_contacts.loc[sorted_contacts["Nơi sinh"] == province].copy()
    group["STT"] = range(1, len(group) + 1)
    sheet_name = safe_sheet_name(province, used)
    sheet_mapping[province] = sheet_name
    write_table(contact_wb.create_sheet(sheet_name), group, f"DANH BẠ — {province}", f"Province{i}")
contact_wb.save(OUT / "contact.xlsx")
print("Nơi sinh → sheet:", sheet_mapping)

## 8. Kiểm tra và tải kết quả
Kiểm tra số người, tổng thống kê, hai biểu đồ, định dạng ngày, giá trị các sheet được giữ lại và số dòng từng sheet nơi sinh. Tuổi được lưu theo ngày chốt, không dùng công thức `TODAY()` để tránh lệch với số liệu biểu đồ khi mở vào ngày khác.

In [ ]:
result = load_workbook(OUT / "data.xlsx")
groups = load_workbook(OUT / "contact.xlsx")
assert result["sorted_contacts"].max_row - 4 == len(sorted_contacts)
assert int(province_counts.sum()) == int(age_counts.sum()) == len(sorted_contacts)
assert len(result["statistics"]._charts) == 2
for name in kept_names:
    assert tuple(result[name].values) == kept_values[name], f"Sheet {name} đã đổi dữ liệu"
birth_col = list(sorted_contacts.columns).index("Ngày sinh") + 1
for row in range(5, len(sorted_contacts) + 5):
    cell = result["sorted_contacts"].cell(row, birth_col)
    assert cell.number_format == "dd-mm-yyyy" and isinstance(cell.value, datetime)
assert sum(ws.max_row - 4 for ws in groups) == len(sorted_contacts)
for province, sheet_name in sheet_mapping.items():
    assert groups[sheet_name].max_row - 4 == int(province_counts.loc[province])
print("✓ Kiểm tra thành công. Đã tạo data.xlsx và contact.xlsx.")

zip_path = OUT / "Bai05_ket_qua.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in ("data.xlsx", "contact.xlsx"):
        archive.write(OUT / name, arcname=name)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Kết quả được lưu tại:", OUT.resolve())